In [1]:
# ============================================================
# HAM10000 - TRAIN REMAINING 4 MODELS
# ResNet50
# EfficientNet-B0
# MobileNetV3-Large
# Swin Transformer-Tiny
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import copy
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader

from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 3. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("DEVICE INFORMATION")
print("=" * 70)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 4. PATHS
# ============================================================

BASE_DIR = Path("..")

DATASET_DIR = BASE_DIR / "dataset"

TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"

SPLITS_DIR = BASE_DIR / "splits"

MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("\nPaths loaded successfully.")

print("Train:", TRAIN_DIR)
print("Validation:", VAL_DIR)
print("Models:", MODELS_DIR)
print("Results:", RESULTS_DIR)


# ============================================================
# 5. CONFIGURATION
# ============================================================

NUM_CLASSES = 7

IMAGE_SIZE = 224

BATCH_SIZE = 32

NUM_WORKERS = 4

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 25

PATIENCE = 5

PIN_MEMORY = torch.cuda.is_available()

print("\nConfiguration:")
print("Classes:", NUM_CLASSES)
print("Batch Size:", BATCH_SIZE)
print("Learning Rate:", LEARNING_RATE)
print("Max Epochs:", MAX_EPOCHS)
print("Patience:", PATIENCE)


# ============================================================
# 6. TRANSFORMS
# ============================================================

train_transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomVerticalFlip(p=0.5),

    transforms.RandomRotation(degrees=20),

    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.05
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


val_transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# 7. DATASETS
# ============================================================

train_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=VAL_DIR,
    transform=val_transform
)

print("\nDataset Information:")
print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))

print("\nClasses:")
print(train_dataset.classes)

assert len(train_dataset) == 8017
assert len(val_dataset) == 1998
assert train_dataset.class_to_idx == val_dataset.class_to_idx


# ============================================================
# 8. DATALOADERS
# ============================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY
)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY
)


print("\nTrain batches:", len(train_loader))
print("Validation batches:", len(val_loader))


# ============================================================
# 9. LOAD CLASS WEIGHTS
# ============================================================

weights_df = pd.read_csv(
    SPLITS_DIR / "class_weights.csv"
)

weights_df = (
    weights_df
    .set_index("Class")
    .loc[train_dataset.classes]
    .reset_index()
)

print("\nClass Weights:")
print(weights_df)


class_weights = torch.tensor(

    weights_df["Class_Weight"].values,

    dtype=torch.float32

).to(device)


# ============================================================
# 10. TRAIN ONE EPOCH FUNCTION
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):

    model.train()

    running_loss = 0.0

    all_predictions = []
    all_labels = []


    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )


        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()


        running_loss += (
            loss.item() * images.size(0)
        )


        predictions = torch.argmax(
            outputs,
            dim=1
        )


        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )


    epoch_loss = (
        running_loss / len(loader.dataset)
    )


    epoch_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )


    epoch_balanced_accuracy = balanced_accuracy_score(
        all_labels,
        all_predictions
    )


    epoch_precision = precision_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    epoch_recall = recall_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    epoch_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    return (

        epoch_loss,

        epoch_accuracy,

        epoch_balanced_accuracy,

        epoch_precision,

        epoch_recall,

        epoch_f1
    )


# ============================================================
# 11. VALIDATION FUNCTION
# ============================================================

def validate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0

    all_predictions = []
    all_labels = []


    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )


            outputs = model(images)


            loss = criterion(
                outputs,
                labels
            )


            running_loss += (
                loss.item() * images.size(0)
            )


            predictions = torch.argmax(
                outputs,
                dim=1
            )


            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )


    epoch_loss = (
        running_loss / len(loader.dataset)
    )


    epoch_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )


    epoch_balanced_accuracy = balanced_accuracy_score(
        all_labels,
        all_predictions
    )


    epoch_precision = precision_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    epoch_recall = recall_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    epoch_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    return (

        epoch_loss,

        epoch_accuracy,

        epoch_balanced_accuracy,

        epoch_precision,

        epoch_recall,

        epoch_f1,

        all_labels,

        all_predictions
    )


# ============================================================
# 12. CREATE MODEL FUNCTION
# ============================================================

def create_model(model_name):


    # --------------------------------------------------------
    # RESNET50
    # --------------------------------------------------------

    if model_name == "ResNet50":

        model = models.resnet50(

            weights=models.ResNet50_Weights.DEFAULT

        )

        num_features = model.fc.in_features

        model.fc = nn.Sequential(

            nn.Dropout(p=0.3),

            nn.Linear(
                num_features,
                NUM_CLASSES
            )
        )


    # --------------------------------------------------------
    # EFFICIENTNET-B0
    # --------------------------------------------------------

    elif model_name == "EfficientNet_B0":

        model = models.efficientnet_b0(

            weights=models.EfficientNet_B0_Weights.DEFAULT

        )

        num_features = model.classifier[1].in_features

        model.classifier = nn.Sequential(

            nn.Dropout(p=0.3),

            nn.Linear(
                num_features,
                NUM_CLASSES
            )
        )


    # --------------------------------------------------------
    # MOBILENET V3 LARGE
    # --------------------------------------------------------

    elif model_name == "MobileNetV3_Large":

        model = models.mobilenet_v3_large(

            weights=models.MobileNet_V3_Large_Weights.DEFAULT

        )

        num_features = model.classifier[3].in_features

        model.classifier[3] = nn.Linear(

            num_features,

            NUM_CLASSES
        )


    # --------------------------------------------------------
    # SWIN TRANSFORMER TINY
    # --------------------------------------------------------

    elif model_name == "Swin_Tiny":

        model = models.swin_t(

            weights=models.Swin_T_Weights.DEFAULT

        )

        num_features = model.head.in_features

        model.head = nn.Sequential(

            nn.Dropout(p=0.3),

            nn.Linear(
                num_features,
                NUM_CLASSES
            )
        )


    else:

        raise ValueError(
            f"Unknown model: {model_name}"
        )


    return model


# ============================================================
# 13. COUNT TRAINABLE PARAMETERS
# ============================================================

def count_trainable_parameters(model):

    return sum(

        parameter.numel()

        for parameter in model.parameters()

        if parameter.requires_grad
    )


# ============================================================
# 14. COMPLETE TRAINING FUNCTION
# ============================================================

def train_complete_model(model_name):


    print("\n")
    print("=" * 80)
    print(f"STARTING TRAINING: {model_name}")
    print("=" * 80)


    # --------------------------------------------------------
    # CREATE MODEL
    # --------------------------------------------------------

    model = create_model(
        model_name
    )

    model = model.to(
        device
    )


    trainable_parameters = count_trainable_parameters(
        model
    )


    print(
        f"\nTrainable Parameters: "
        f"{trainable_parameters:,}"
    )


    # --------------------------------------------------------
    # LOSS
    # --------------------------------------------------------

    criterion = nn.CrossEntropyLoss(

        weight=class_weights

    )


    # --------------------------------------------------------
    # OPTIMIZER
    # --------------------------------------------------------

    optimizer = optim.AdamW(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY

    )


    # --------------------------------------------------------
    # SCHEDULER
    # --------------------------------------------------------

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(

        optimizer,

        mode="max",

        factor=0.5,

        patience=2

    )


    # --------------------------------------------------------
    # HISTORY
    # --------------------------------------------------------

    history = {

        "train_loss": [],
        "val_loss": [],

        "train_accuracy": [],
        "val_accuracy": [],

        "train_balanced_accuracy": [],
        "val_balanced_accuracy": [],

        "train_precision": [],
        "val_precision": [],

        "train_recall": [],
        "val_recall": [],

        "train_f1": [],
        "val_f1": [],

        "learning_rate": []

    }


    best_val_f1 = -float("inf")

    epochs_without_improvement = 0


    best_model_path = (

        MODELS_DIR /

        f"{model_name.lower()}_best.pth"

    )


    # ========================================================
    # TRAINING LOOP
    # ========================================================

    for epoch in range(MAX_EPOCHS):


        start_time = time.time()


        # TRAIN

        (

            train_loss,

            train_acc,

            train_bal_acc,

            train_precision,

            train_recall,

            train_f1

        ) = train_one_epoch(

            model,

            train_loader,

            criterion,

            optimizer,

            device

        )


        # VALIDATION

        (

            val_loss,

            val_acc,

            val_bal_acc,

            val_precision,

            val_recall,

            val_f1,

            val_labels,

            val_predictions

        ) = validate(

            model,

            val_loader,

            criterion,

            device

        )


        scheduler.step(
            val_f1
        )


        current_lr = (

            optimizer.param_groups[0]["lr"]

        )


        # SAVE HISTORY

        history["train_loss"].append(
            train_loss
        )

        history["val_loss"].append(
            val_loss
        )


        history["train_accuracy"].append(
            train_acc
        )

        history["val_accuracy"].append(
            val_acc
        )


        history["train_balanced_accuracy"].append(
            train_bal_acc
        )

        history["val_balanced_accuracy"].append(
            val_bal_acc
        )


        history["train_precision"].append(
            train_precision
        )

        history["val_precision"].append(
            val_precision
        )


        history["train_recall"].append(
            train_recall
        )

        history["val_recall"].append(
            val_recall
        )


        history["train_f1"].append(
            train_f1
        )

        history["val_f1"].append(
            val_f1
        )


        history["learning_rate"].append(
            current_lr
        )


        epoch_time = (
            time.time() - start_time
        )


        # PRINT

        print("\n" + "-" * 80)

        print(
            f"{model_name} | "
            f"Epoch [{epoch + 1}/{MAX_EPOCHS}]"
        )

        print(
            f"Time: {epoch_time:.1f}s"
        )

        print("-" * 80)

        print(
            f"Train Loss: {train_loss:.4f} | "
            f"Acc: {train_acc:.4f} | "
            f"Precision: {train_precision:.4f} | "
            f"Recall: {train_recall:.4f} | "
            f"F1: {train_f1:.4f}"
        )

        print(
            f"Val Loss: {val_loss:.4f} | "
            f"Acc: {val_acc:.4f} | "
            f"Precision: {val_precision:.4f} | "
            f"Recall: {val_recall:.4f} | "
            f"F1: {val_f1:.4f}"
        )

        print(
            f"Learning Rate: {current_lr:.6f}"
        )


        # ----------------------------------------------------
        # SAVE BEST MODEL
        # ----------------------------------------------------

        if val_f1 > best_val_f1:


            best_val_f1 = val_f1


            torch.save(

                model.state_dict(),

                best_model_path

            )


            epochs_without_improvement = 0


            print(
                f"✓ BEST MODEL SAVED "
                f"| Macro F1: {best_val_f1:.4f}"
            )


        else:


            epochs_without_improvement += 1


            print(
                f"No improvement: "
                f"{epochs_without_improvement}/{PATIENCE}"
            )


        # ----------------------------------------------------
        # EARLY STOPPING
        # ----------------------------------------------------

        if epochs_without_improvement >= PATIENCE:


            print(
                "\nEARLY STOPPING TRIGGERED"
            )

            break


    # ========================================================
    # RESTORE BEST MODEL
    # ========================================================

    model.load_state_dict(

        torch.load(

            best_model_path,

            map_location=device

        )

    )


    model.eval()


    # ========================================================
    # FINAL VALIDATION
    # ========================================================

    (

        final_val_loss,

        final_val_acc,

        final_val_bal_acc,

        final_val_precision,

        final_val_recall,

        final_val_f1,

        final_labels,

        final_predictions

    ) = validate(

        model,

        val_loader,

        criterion,

        device

    )


    # ========================================================
    # SAVE HISTORY
    # ========================================================

    history_df = pd.DataFrame(
        history
    )


    history_path = (

        MODELS_DIR /

        f"{model_name.lower()}_history.csv"

    )


    history_df.to_csv(

        history_path,

        index=False

    )


    # ========================================================
    # PLOT LOSS
    # ========================================================

    epochs = range(

        1,

        len(history["train_loss"]) + 1

    )


    plt.figure(figsize=(8, 5))

    plt.plot(
        epochs,
        history["train_loss"],
        label="Train Loss"
    )

    plt.plot(
        epochs,
        history["val_loss"],
        label="Validation Loss"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")

    plt.title(
        f"{model_name} - Training and Validation Loss"
    )

    plt.legend()

    plt.grid()

    plt.show()


    # ========================================================
    # PLOT MACRO F1
    # ========================================================

    plt.figure(figsize=(8, 5))

    plt.plot(
        epochs,
        history["train_f1"],
        label="Train Macro F1"
    )

    plt.plot(
        epochs,
        history["val_f1"],
        label="Validation Macro F1"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")

    plt.title(
        f"{model_name} - Macro F1 Score"
    )

    plt.legend()

    plt.grid()

    plt.show()


    # ========================================================
    # MODEL SUMMARY
    # ========================================================

    print("\n")
    print("=" * 80)
    print(f"{model_name} COMPLETED")
    print("=" * 80)

    print(
        "Accuracy:",
        f"{final_val_acc:.4f}"
    )

    print(
        "Balanced Accuracy:",
        f"{final_val_bal_acc:.4f}"
    )

    print(
        "Macro Precision:",
        f"{final_val_precision:.4f}"
    )

    print(
        "Macro Recall:",
        f"{final_val_recall:.4f}"
    )

    print(
        "Macro F1:",
        f"{final_val_f1:.4f}"
    )

    print(
        "Trainable Parameters:",
        f"{trainable_parameters:,}"
    )


    # ========================================================
    # RETURN RESULTS
    # ========================================================

    result = {

        "Model": model_name,

        "Accuracy": final_val_acc,

        "Balanced Accuracy": final_val_bal_acc,

        "Precision": final_val_precision,

        "Recall": final_val_recall,

        "F1-Score": final_val_f1,

        "Trainable Parameters": trainable_parameters,

        "Epochs Completed": len(
            history["train_loss"]
        ),

        "Best Model Path": str(
            best_model_path
        )

    }


    # FREE GPU MEMORY

    del model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return result


# ============================================================
# 15. LIST OF REMAINING MODELS
# ============================================================

MODEL_LIST = [

    "ResNet50",

    "EfficientNet_B0",

    "MobileNetV3_Large",

    "Swin_Tiny"

]


# ============================================================
# 16. TRAIN ALL MODELS AUTOMATICALLY
# ============================================================

all_results = []


for model_name in MODEL_LIST:


    result = train_complete_model(
        model_name
    )


    all_results.append(
        result
    )


# ============================================================
# 17. ADD YOUR EXISTING DENSENET121 RESULT
# ============================================================

densenet_result = {

    "Model": "DenseNet121",

    "Accuracy": 0.8453,

    "Balanced Accuracy": 0.7659,

    "Precision": np.nan,

    "Recall": np.nan,

    "F1-Score": 0.7557,

    "Trainable Parameters": np.nan,

    "Epochs Completed": 25,

    "Best Model Path":
        str(
            MODELS_DIR /
            "densenet121_baseline_best.pth"
        )

}


all_results.append(
    densenet_result
)


# ============================================================
# 18. CREATE FINAL COMPARISON TABLE
# ============================================================

comparison_df = pd.DataFrame(
    all_results
)


# Put DenseNet first

model_order = [

    "DenseNet121",

    "ResNet50",

    "EfficientNet_B0",

    "MobileNetV3_Large",

    "Swin_Tiny"

]


comparison_df["Model"] = pd.Categorical(

    comparison_df["Model"],

    categories=model_order,

    ordered=True

)


comparison_df = comparison_df.sort_values(
    "Model"
)


# Convert metrics to percentage

for column in [

    "Accuracy",

    "Balanced Accuracy",

    "Precision",

    "Recall",

    "F1-Score"

]:

    comparison_df[column] = (
        comparison_df[column] * 100
    ).round(2)


print("\n")
print("=" * 100)
print("FINAL MODEL COMPARISON")
print("=" * 100)

print(comparison_df)


# ============================================================
# 19. SAVE FINAL RESULTS
# ============================================================

comparison_path = (

    RESULTS_DIR /

    "final_5_model_comparison.csv"

)


comparison_df.to_csv(

    comparison_path,

    index=False

)


print("\n")
print("=" * 100)
print("ALL 5 MODELS COMPLETED")
print("=" * 100)

print(
    "\nComparison table saved at:"
)

print(
    comparison_path
)

DEVICE INFORMATION
Device: cuda
GPU: NVIDIA GeForce RTX 5050 Laptop GPU

Paths loaded successfully.
Train: ../dataset/train
Validation: ../dataset/val
Models: ../models
Results: ../results

Configuration:
Classes: 7
Batch Size: 32
Learning Rate: 0.0001
Max Epochs: 25
Patience: 5

Dataset Information:
Train images: 8017
Validation images: 1998

Classes:
['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

Train batches: 251
Validation batches: 63

Class Weights:
   Class  Image_Count  Percentage  Class_Weight
0  akiec          257        3.21        4.4564
1    bcc          412        5.14        2.7798
2    bkl          891       11.11        1.2854
3     df           86        1.07       13.3173
4    mel          890       11.10        1.2868
5     nv         5367       66.95        0.2134
6   vasc          114        1.42       10.0464


STARTING TRAINING: ResNet50

Trainable Parameters: 23,522,375


[W1005 18:26:45.092193869 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 102760448 bytes (free: 34340864, total: 8082096128).
[W1005 18:26:45.092272290 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 102760448 bytes (free: 34340864, total: 8082096128).


OutOfMemoryError: CUDA out of memory. Tried to allocate 98.00 MiB. GPU 0 has a total capacity of 7.53 GiB of which 32.75 MiB is free. Process 13444 has 322.00 MiB memory in use. Process 13575 has 674.00 MiB memory in use. Process 15044 has 804.00 MiB memory in use. Process 15259 has 4.85 GiB memory in use. Including non-PyTorch memory, this process has 862.00 MiB memory in use. Of the allocated memory 672.85 MiB is allocated by PyTorch, and 81.15 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)